# Pipeline stage 1: the search for phage-encoded acetyltransferases

### Stage 1.1: input phages & proteins

**Goals:**
* Extract all unique phages from the Virus-Host database overview file.
* Fetch protein information from NCBI protein for all these phages.
* Create overview & analyze taxonomy of these phages and their hosts.
* Extract all annotated acetyltransferases from these phages.
* Extract all proteins of unknown function from these phages for further mining of (unidentified) acetyltransferases.

**Requires:**
*  virushostdb_host_pseudomonas_25feb2024.tsv stored in associated_data : results of search for '*Pseudomonas*' as host in the [Virus-Host database](https://www.genome.jp/virushostdb/) (version: February 25th, 2024)

**Generates:**
* phage_data.tsv : a tab-seperated phage-centric overview of the virus & host taxonomic information from virushostdb_host_pseudomonas_25feb2024.tsv, summarizing all included phages (one phage per line) by their NCBI taxonomy identifier (virus_taxid), name as reported in the Virus-Host database (virus_name), taxonomic lineage as reported in the Virus-Host database (virus_lineage), the shortened name used to refer to the virus in this study (short_name), the combined shortened name and taxonomy identifier used to reproducibily refer to the virus in this study (name_taxid), a list of the host(s) of the virus as reported in the Virus-Host database (host_name) and a list of the taxonomic lineage(s) of the host(s) of the virus as reported in the Virus-Host database (host_lineage)
* in folder protein_overview:
    * PHAGE_protein_overview.tsv : a tab-seperated overview of the NCBI proteins for each PHAGE, listing NCBI protein unique identifier (ncbi_protein_uid), NCBI protein name (ncbi_protein_name) and protein length in amino acids (ncbi_protein_length), with each row representing one NCBI protein entry
    * protein_overview.tsv : a tab-seperated aggregated overview of the NCBI proteins for all phages, formatted similarly to PHAGE_protein_overview.tsv, but with an additional first column indicating the PHAGE (name_taxid)
    * annotated_acetyltransferases_overview.tsv : a tab-seperated overview of the NCBI and UniProt proteins of our phages that contain 'GNAT' and/or 'acetyltransferase' in their protein name, with each row representing one protein (column names as before, with the addition of 'sequence' for the amino acid sequence of proteins, 'length' instead of 'ncbi_protein_length', 'uniprot_protein_id' for UniProt identifiers, and 'uniprot_protein_name' for UniProt protein names). Each row represents a unique phage-protein pairing. 
    * PHAGE_input_search.tsv : a tab-seperated overview of the NCBI proteins that are unannotated and within the expected size range of acetyltransferases for each PHAGE, with each row representing one unique (based on sequence) NCBI protein entry (column names as before)
    * input_search.tsv : a tab-seperated aggregated overview of the NCBI proteins that are unannotated and within the expected size range of acetyltransferases for all phages, formatted similarly to PHAGE_input_search.tsv, but with an additional first column indicating the PHAGE (name_taxid)

#### General settings, imports, variables and environments

Conda environment: viral_act_input

Created with `conda create -n viral_act_input python=3.10`.
Then installed `jupyter notebook`, `pandas` through conda (using command `conda install`) & `more_iterools` through pip (command below).

In [ ]:
!pip install more-itertools

In [1]:
!conda list --explicit

# This file may be used to create an environment using:
# $ conda create --name <env> --file <this file>
# platform: win-64
# created-by: conda 24.11.3
@EXPLICIT
https://conda.anaconda.org/conda-forge/noarch/ca-certificates-2025.10.5-h4c7d964_0.conda
https://conda.anaconda.org/conda-forge/noarch/python_abi-3.10-8_cp310.conda
https://conda.anaconda.org/conda-forge/noarch/tzdata-2025b-h78e105d_0.conda
https://conda.anaconda.org/conda-forge/win-64/ucrt-10.0.26100.0-h57928b3_0.conda
https://conda.anaconda.org/conda-forge/win-64/winpty-0.4.3-4.tar.bz2
https://conda.anaconda.org/conda-forge/win-64/libwinpthread-12.0.0.r4.gg4f2fc60ca-h57928b3_10.conda
https://conda.anaconda.org/conda-forge/win-64/vcomp14-14.44.35208-h818238b_32.conda
https://conda.anaconda.org/conda-forge/win-64/vc14_runtime-14.44.35208-h818238b_32.conda
https://conda.anaconda.org/conda-forge/win-64/vc-14.3-h2b53caa_32.conda
https://conda.anaconda.org/conda-forge/win-64/bzip2-1.0.8-h0ad9c76_8.conda
https://conda.anaconda.org/

In [2]:
# imports
import json
import os
import requests
import pandas as pd

from io import StringIO
from more_itertools import batched

In [3]:
# settings for requests
sess = requests.Session()
adapter = requests.adapters.HTTPAdapter(max_retries = 10)
sess.mount("https://", adapter)

In [4]:
# clear reference to different directories
pipeline_search_dir = os.getcwd()
master_dir = os.path.abspath(os.path.join(pipeline_search_dir, os.pardir, os.pardir))

In [ ]:
# creating a directory for all the data we will generate
os.mkdir(os.path.join(pipeline_search_dir, "a_input"))

#### Goal 1: extract all unique phages from the Virus-Host database overview file

The overview downloaded from the Virus-Host database contains 1 line per virus-host pair, meaning that phages with multiple hosts are represented multiple times. Here, we create a non-redundant overview of the different phages.

In [5]:
# reading in the data
virushost_data = pd.read_table(os.path.join(master_dir, "associated_data", "virushostdb_host_pseudomonas_25feb2024.tsv"))

In [6]:
# creating phage centered table
phage_data = virushost_data[["virus tax id", "virus name", "virus lineage"]]
phage_data = phage_data.drop_duplicates()
phage_data = phage_data.reset_index(drop = True)

In [7]:
# function to shorten phage names for easier use
def rename_phage(name):
    new_name = name
    if "virus" in new_name.lower():
        new_name = new_name.split("virus")[1].strip()
    if "phage" in new_name.lower():
        if "phage sp." in new_name.lower():
            new_name = new_name.split("phage sp.")[1].strip()
        else:
            new_name = new_name.split("phage")[1].strip()
    new_name = new_name.replace(" ", "_")
    return new_name

In [8]:
# updating the phage table 
    # easier names
phage_data["short_name"] = phage_data.apply(lambda row : rename_phage(row["virus name"]), axis = 1)
phage_data["name_taxid"] = phage_data["short_name"] + "_taxid_" + phage_data["virus tax id"].astype(str)
    # adding aggregated host data
agg_host_data = virushost_data.groupby(["virus tax id", "virus name"], as_index = False).agg({"host name" : list, "host lineage" : list})
phage_data = phage_data.merge(agg_host_data)
phage_data = phage_data.rename(columns = {"virus tax id" : "virus_taxid", "virus name" : "virus_name", "virus lineage" : "virus_lineage",
                                          "host name": "host_name", "host lineage" : "host_lineage"})

#### Goal 2: fetch protein information from NCBI protein for all these phages

To fetch protein information, we will create a list of all phages in our dataset. 

In [9]:
# generating a list of our phages
phages = list(phage_data["name_taxid"]) 
# sadly some phage names are computationally challenging, containing / or . 
    # let's make a dict of those with the computationally easier alternative
annoying_phage_names_dict = {}
for phage in phages:
    if "." in phage:
        annoying_phage_names_dict[phage] = phage.replace(".", "_")
    if "/" in phage:
        annoying_phage_names_dict[phage] = phage.replace("/", "_")

Then, we can use eutils to obtain a summary of the proteins for all our phages. *This was executed on February 26th, 2024, and not repeated here, as more proteins might have been added to NCBI protein since then, which could potentially affect downstream reuslts.*

In [ ]:
# creating a directory for all the data we will generate
os.mkdir(os.path.join(pipeline_search_dir, "a_input", "protein_overview"))

In [ ]:
# the URLs
url_overview = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=protein&term=txid{0}&retmax=1000&retmode=json"
url_proteins = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=protein&id={0}&retmax=1000&retmode=json"
# We create 1 file with all proteins, and then a file per phage
f_all_phages = open(os.path.join(pipeline_search_dir, "a_input", "protein_overview", "protein_overview.tsv"), "w")
f_all_phages.write("name_taxid\tncbi_protein_uid\tncbi_protein_name\tncbi_protein_length\n")
for phage in phages:
    taxid = phage.split("taxid_")[1]
    phage_protein_ids = sess.get(url_overview.format(taxid), stream = True).text
    data_ids = json.loads(phage_protein_ids)
    # check if any proteins present for this phage
    if len(data_ids.get("esearchresult").get("idlist")) != 0:
        if phage in annoying_phage_names_dict.keys():
            f_per_phage = open(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{annoying_phage_names_dict.get(phage)}_protein_overview.tsv"), "w")
        else:    
            f_per_phage = open(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_protein_overview.tsv"), "w")
        # batching to circumvent too long URLs
        f_per_phage.write("ncbi_protein_uid\tncbi_protein_name\tncbi_protein_length\n")
        for batch in batched(data_ids.get("esearchresult").get("idlist"), 100):
            phage_protein_ids_string = ",".join(batch)
            response = sess.get(url_proteins.format(phage_protein_ids_string), stream = True)
            response.raise_for_status()
            phage_protein_overview = response.text
            data = json.loads(phage_protein_overview)
            ids = data.get("result").get("uids")
            for id in ids:
                name = data.get("result").get(id).get("title")
                length = data.get("result").get(id).get("slen")
                f_per_phage.write(f"{id}\t{name}\t{length}\n")
                f_all_phages.write(f"{phage}\t{id}\t{name}\t{length}\n")
        f_per_phage.close()
f_all_phages.close()

Let's read in the data as obtained in February 2024, and use it for the rest of notebook: 

In [10]:
# remove phages with no proteins in NCBI protein from our list
protein_data = pd.read_table(os.path.join(pipeline_search_dir, "a_input", "protein_overview", "protein_overview.tsv"))
removed = []
# find missing phages
for phage in phages:
    if phage not in protein_data["name_taxid"].unique():
        removed.append(phage)
# remove from list
for item in removed:
    phages.remove(item)
# print out number of removed phages
num_removed = len(removed)
print(f"Removed {num_removed} phages from the dataset, as they did not have any NCBI protein entries fetched.")
# effectively remove them from the dataset
phage_data = phage_data[phage_data["name_taxid"].isin(phages)]
phage_data = phage_data.reset_index(drop = True)

Removed 73 phages from the dataset, as they did not have any NCBI protein entries fetched.


In [11]:
# store overview of phages 
phage_data.to_csv(os.path.join(pipeline_search_dir, "a_input", "phage_data.tsv"), sep = "\t", index = False)

#### Goal 3: create overview & analyze taxonomy of these phages and their hosts

In achieving goals 1 & 2, we have created an overview of the different hosts and phages in this study in a phage-centric manner. Now, we will summarize this data to better grasp its contents, and gain understanding into the phages in our dataset.

First, we will summarize the **virus taxonomy**. This is not straightforward, as not all taxonomy levels are necessarily filled (e.g.: jumping from class to subfamily, omitting order and family). Briefly, the 'virus_lineage' was extracted from the phage data, and programatically processed to represent all taxonomic levels of interest. 

In [12]:
# function to assign all taxonomic levels up until genus (incl. None values for unmentioned levels), 
    # and return a dictionary with those taxonomic groupings, with as keys the textual descriptors used to distinguish the taxonomic levels
def assign_virus_taxonomy(taxonomy_str):
    tdomain, trealm, tkingdom, tphylum, tclass, torder, tfamily, tgenus = [None] * 8
    tax_dict = {"Viruses" : tdomain, "viria" : trealm, "virae" : tkingdom, "viricota" : tphylum, 
                "viricetes" : tclass, "virales" : torder, "viridae" : tfamily, "virus" : tgenus}
    taxonomy_list = taxonomy_str.split(";")
    for item in taxonomy_list:
        for key, value in tax_dict.items():
            # (value == None) makes sure that more specific descriptors (from lower taxonomic levels) don't override initial assignments, 
                # as the string used as input goes from high to low level of taxonomy
            # (len(item.split()) == 1) makes sure we don't assign species names (or lower taxonomic levels) to higher taxonomic levels, when these are unassigned
                # as the species level is the first level making use of two words
            if (key in item) and (value == None) and (len(item.split()) == 1):
                tax_dict[key] = item.strip()
    return tax_dict

In [13]:
# extract taxonomy information
virus_tax = phage_data["virus_lineage"]
# empty list to store taxonomy dicts for each virus
all_tax = list()
# filling the list of with all the taxonomy dicts
for taxonomy in virus_tax:
    all_tax.append(assign_virus_taxonomy(taxonomy))
# converting the information to a dataframe, renaming the columns to correctly name the taxonomic levels, and storing the information
all_tax_df = pd.DataFrame(all_tax)
all_tax_df = all_tax_df.rename(columns = {"Viruses": "Virus", "viria": "Realm", "virae" : "Kingdom", "viricota" : "Phylum", 
                                          "viricetes" : "Class", "virales" : "Order", "viridae": "Family", "virus" : "Genus"})
# printing a summary per level
tax_summary = {tax_level: all_tax_df[tax_level].value_counts() for tax_level in all_tax_df.columns}
# display the summary
for tax_level, counts in tax_summary.items():
    print(f"\nColumn '{tax_level}' value counts:")
    print(counts)


Column 'Virus' value counts:
Virus
Viruses    887
Name: count, dtype: int64

Column 'Realm' value counts:
Realm
Duplodnaviria    869
Riboviria         10
Monodnaviria       5
Varidnaviria       1
Name: count, dtype: int64

Column 'Kingdom' value counts:
Kingdom
Heunggongvirae    869
Orthornavirae      10
Loebvirae           5
Bamfordvirae        1
Name: count, dtype: int64

Column 'Phylum' value counts:
Phylum
Uroviricota          869
Duplornaviricota       7
Hofneiviricota         5
Lenarviricota          3
Preplasmiviricota      1
Name: count, dtype: int64

Column 'Class' value counts:
Class
Caudoviricetes      869
Vidaverviricetes      7
Faserviricetes        5
Leviviricetes         3
Tectiliviricetes      1
Name: count, dtype: int64

Column 'Order' value counts:
Order
Mindivirales     7
Tubulavirales    5
Norzivirales     3
Kalamavirales    1
Name: count, dtype: int64

Column 'Family' value counts:
Family
Autographiviridae      139
Schitoviridae           45
Mesyanzhinovviridae   

Summarizing: 98% of all phages are dsDNA phages, which are all Caudoviricites (or Varidnaviriae). There are 5 Monodnaviriae, 10 Riboviriae and 2 unclassified bacterial viruses. Of the dsDNA phages, 16% falls under the Autographviridae family, and 18% under the Pbunavirus genus; no other taxonomic groups (except unclassified Caudoviricetes) exceed 10%.

Next, we look into the **host taxonomy**. We create an overview of the different hosts and how abundant they are amongst our phages by aggregating the data by the host name. Let's first look at how many phages are reported to have multiple hosts:

In [14]:
# joining entries with multiple hosts
phage_data["host_name_string"] = [",".join(map(str, host)) for host in phage_data["host_name"]]
phage_data["host_lineage_string"] = [",".join(map(str, host)) for host in phage_data["host_lineage"]]
host_data = phage_data.groupby(["host_name_string","host_lineage_string"], as_index = False).size() 

In [15]:
# Some statistics
    # number of phages with more than 1 host 
num_multiple_host = host_data[host_data["host_name_string"].str.contains(r",")]["size"].sum()
    # number of phages with just 1 host 
num_one_host = host_data["size"].sum() - num_multiple_host
    # of those with multiple hosts, number with species diversity
multiple_host_names = list(host_data[host_data["host_name_string"].str.contains(r",")]["host_name_string"])
multiple_host_multiple_species = []
for hostset in multiple_host_names:
    list_hosts = hostset.split(",")
    species_set = set()
    for host in list_hosts:
        species = host.split()
        species_set.add(species[0])
    if len(species_set) > 1:
        multiple_host_multiple_species.append(hostset)
subset_multiple_hosts = host_data["host_name_string"].isin(multiple_host_multiple_species)
num_multiple_host_multiple_species = host_data[subset_multiple_hosts]["size"].sum()
    # of those with multiple hosts, number with strain diversity
num_multiple_host_within_species = num_multiple_host - num_multiple_host_multiple_species
# print results
print("Overview host taxonomy statistics:")
print("Phages with only 1 host: ", num_one_host)
print("Phages with more than 1 host: ", num_multiple_host)
print("Of those with multiple hosts...")
print("\t", num_multiple_host_multiple_species, "phages have multiple hosts that are from different species.")
print("\t", num_multiple_host_within_species, "phages have multiple hosts that are from the same species.")

Overview host taxonomy statistics:
Phages with only 1 host:  848
Phages with more than 1 host:  39
Of those with multiple hosts...
	 6 phages have multiple hosts that are from different species.
	 33 phages have multiple hosts that are from the same species.


To look into the host taxonomy in more detail, we use the same strategy as for the virus taxonomy. To make this overview biologically sensible, we weight the different taxonomies by the amount of phages they were found in (so total: 887, like the virus taxonomy analysis). In order to not exceed the total number of phages, phages for which multiple hosts are reported are split proportionally across these different hosts.

In [16]:
# ugly solution for splitting multiple hosts proportionally (definitely not most computationally efficient)
    # seperate out the entries with multiple hosts in a different dataframe
host_data_multi = host_data[host_data["host_name_string"].str.contains(r",")].reset_index(drop = True)
    # now loop over those entries, and create lists with the different host name and lineages + calculate count per host by proportional split
for index, row in host_data_multi.iterrows():
    count = row["host_name_string"].count(",") + 1
    host_name_list = list(row["host_name_string"].split(","))
    host_tax_list = list(row["host_lineage_string"].split(","))
    count_adapted = int(row["size"])/count
    # now actually add entries to the table where the multi-host rows are split into seperate row
    for index, entry in enumerate(host_name_list):
        host_data_multi = pd.concat([host_data_multi, pd.DataFrame({"host_name_string" : [entry], "host_lineage_string" : [host_tax_list[index]], 
                                                                    "size" : [count_adapted]})], ignore_index = True)
    # once finished, we can drop all rows with multiple hosts
host_data_multi = host_data_multi[~host_data_multi["host_name_string"].str.contains(r",")].reset_index(drop = True)
    # now we can combine this with the single host entries
host_data_one = host_data[~host_data["host_name_string"].str.contains(r",")].reset_index(drop = True)
host_data_combined = pd.concat([host_data_one, host_data_multi], ignore_index = True)
host_data_combined = host_data_combined.groupby("host_name_string", as_index = False).agg({"host_lineage_string" : "first", "size" : "sum"})

Let's now make it easy to work with the different taxonomic level specificities as well, by implementing a similar solution as before:

In [17]:
# correct some pathovar names
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas coronafaciens pv. porri"), "host_name_string"] = "Pseudomonas coronafaciens; Pseudomonas coronafaciens pv. porri"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas savastanoi pv. glycinea"), "host_name_string"] = "Pseudomonas savastanoi; Pseudomonas savastanoi pv. glycinea"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas savastanoi pv. phaseolicola"), "host_name_string"] = "Pseudomonas savastanoi; Pseudomonas savastanoi pv. phaseolicola"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas syringae pv. actinidiae ICMP 18800"), "host_name_string"] = "Pseudomonas syringae; Pseudomonas syringae pv. actinidiae; Pseudomonas syringae pv. actinidiae ICMP 18800"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas syringae pv. actinidiae"), "host_name_string"] = "Pseudomonas syringae; Pseudomonas syringae pv. actinidiae"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas syringae pv. avii"), "host_name_string"] = "Pseudomonas syringae; Pseudomonas syringae pv. syringae" 
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas syringae pv. syringae"), "host_name_string"] = "Pseudomonas coronafaciens; Pseudomonas coronafaciens pv. garcae"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas syringae pv. tomato str. DC3000"), "host_name_string"] = "Pseudomonas syringae group genomosp. 3; Pseudomonas syringae pv. tomato; Pseudomonas syringae pv. tomato str. DC3000"
host_data_combined.loc[host_data_combined["host_name_string"].str.contains("Pseudomonas syringae pv. tomato"), "host_name_string"] = "Pseudomonas syringae group genomosp. 3; Pseudomonas syringae pv. tomato"

In [18]:
# function to assign all taxonomic levels up until genus (incl. None values for unmentioned levels), 
    # and return a dictionary with those taxonomic groupings, with as keys the (textual) descriptors used to distinguish the taxonomic levels
def assign_bac_taxonomy(taxonomy_str):
    tdomain, tkingdom, tphylum, tclass, torder, tfamily, tgenus, tgroup, tspecies = [None] * 9
    # as can be seen from the code, this data does assume Gammaproteobacteria, and might not be effective outside of that class
    tax_dict = {"Bacteria": tdomain, "Pseudomonadati" : tkingdom, "Pseudomonadota" : tphylum, "Gammaproteobacteria" : tclass, 
                "ales" : torder, "aceae" : tfamily}
    taxonomy_list = taxonomy_str.split(";")
    for item in taxonomy_list:
        for key, value in tax_dict.items():
            # same reasoning as before
            if (key in item) and (value == None) and (len(item.split()) == 1):
                tax_dict[key] = item.strip()
        # for all our data, all the levels up to family can be filled based on recognition of specific substrings, 
            # however, for genus this is not the case, but this is the last one word level so we can fill using that
        if None not in tax_dict.items():
            # the final one word taxonomic level is genus
            if len(item.split()) == 1:
                tgenus = item.strip()
                tax_dict["genus"] = tgenus
                # we now also add a None element for group and species
                tax_dict["group"] = tgroup
                tax_dict["species"] = tspecies
            # some descriptions contain a "group" subgenus classification, so let's extract these 
            elif (("group" in item) or ("complex" in item)) and (tgroup == None):
                tgroup = item.strip()
                tax_dict["group"]  = tgroup
            # filling out species, usually the first multi-word descriptor (except for groups)
            elif (len(item.split()) > 1) and (tspecies == None):
                if ("genomosp. 1" in item) or ("genomosp. 2" in item):
                    continue
                else:
                    tspecies = item.strip()
                    tax_dict["species"] = tspecies
    return tax_dict

In [19]:
# combining the most specific naming with the lineage information
host_data_combined["host_lineage_full"] = host_data_combined["host_lineage_string"] + "; " + host_data_combined["host_name_string"]
# extracting the taxonomic information
bac_tax = host_data_combined["host_lineage_full"]
# empty list to store host taxonomy dicts for each virus
all_bac_tax = list()
# filling the host taxonomy list
for taxonomy in bac_tax:
    all_bac_tax.append(assign_bac_taxonomy(taxonomy))
# converting the information to a dataframe, renaming the columns to correctly name the taxonomic levels, 
    # combining this with the weighted sizes of these host levels (taking advantage of ordered lists)
all_bac_tax_df = pd.DataFrame(all_bac_tax)
all_bac_tax_df = all_bac_tax_df.rename(columns = {"Bacteria": "Domain", "Pseudomonadati" : "Kingdom", "Pseudomonadota" : "Phylum", 
                                                  "Gammaproteobacteria" : "Class", "ales" : "Order", "aceae" : "Family", "genus": "Genus", 
                                                  "group" : "Group", "species": "Species"})
all_bac_tax_df["Size_corrected"] = host_data_combined["size"]
# remove duplicate species groups
all_bac_tax_df_sp = all_bac_tax_df.groupby("Species", as_index = False).agg({"Domain": "first", "Kingdom": "first", "Phylum" : "first", "Class" : "first",
                                                                           "Order" : "first", "Family" : "first", "Genus" : "first", "Group" : "first",
                                                                           "Species" : "first", "Size_corrected": "sum"})
all_bac_tax_df_na = all_bac_tax_df[all_bac_tax_df["Species"].isna()]
all_bac_tax_df = pd.concat([all_bac_tax_df_sp, all_bac_tax_df_na], ignore_index = True)
# display a summary per level
bac_tax_summary = {}
for tax_level in ["Domain", "Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Group", "Species"]:
    bac_tax_summary[tax_level] = all_bac_tax_df.groupby(tax_level, as_index = False)["Size_corrected"].sum()
for tax_level, counts in bac_tax_summary.items():
    print(f"\nAggregated by {tax_level}:")
    print(counts)


Aggregated by Domain:
     Domain  Size_corrected
0  Bacteria           887.0

Aggregated by Kingdom:
Empty DataFrame
Columns: [Kingdom, Size_corrected]
Index: []

Aggregated by Phylum:
           Phylum  Size_corrected
0  Pseudomonadota           887.0

Aggregated by Class:
                 Class  Size_corrected
0  Gammaproteobacteria           887.0

Aggregated by Order:
              Order  Size_corrected
0  Enterobacterales        2.208333
1      Moraxellales        0.125000
2   Pseudomonadales      884.541667
3       Vibrionales        0.125000

Aggregated by Family:
               Family  Size_corrected
0  Enterobacteriaceae        2.083333
1       Moraxellaceae        0.125000
2      Morganellaceae        0.125000
3    Pseudomonadaceae      884.541667
4        Vibrionaceae        0.125000

Aggregated by Genus:
           Genus  Size_corrected
0  Acinetobacter        0.125000
1    Escherichia        1.958333
2        Proteus        0.125000
3    Pseudomonas      884.541667
4    

In summary: 64% of phages have *P. aeruginosa* as host, 5% *P. syringae*, 3% *P. fluorescens*, 2% *P. putida* and 2% *P. coronafaciens*. The majority of the other phages have a host described as Pseudomonas, without further detail.

#### Goal 4: extract all annotated acetyltransferases from these phages

Now, we can look into whether these phages have any annotated acetyltransferases. In order to do so, we will use the description from NCBI protein, and look for proteins with 'GNAT' or 'acetyltransferase' in their description. As NCBI protein does contain duplicate entries, we also deduplicate these results. Duplicates are identified on the basis of virus taxonomy & sequence (i.e.: a protein with the same virus taxonomy and the same sequence as another protein is designated as a duplicate, and removed).

In [20]:
# function for getting protein sequence through ncbi eutils
def fetch_sequence_ncbi(protein_id):
    url_fasta = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=protein&id={0}&rettype=fasta"
    # getting info
    fasta = sess.get(url_fasta.format(protein_id), stream = True).text
    seq = fasta.split("\n")
    return "".join(seq[1:-2])

In [21]:
# extracting all annotated acetyltransferases
acetyltransferases_ncbi = protein_data[(protein_data["ncbi_protein_name"].str.contains("GNAT", case = False)) |
    (protein_data["ncbi_protein_name"].str.contains("acetyltransferase", case = False))]
acetyltransferases_ncbi = acetyltransferases_ncbi.reset_index(drop = True)
# fetch protein sequences
acetyltransferases_ncbi["sequence"] = acetyltransferases_ncbi.apply(lambda row: fetch_sequence_ncbi(row["ncbi_protein_uid"]), axis = 1)
# deduplicate based on sequence
acetyltransferases_ncbi = acetyltransferases_ncbi.drop_duplicates(subset = ["name_taxid", "sequence"], keep = "last").reset_index(drop = True)   

So far, we have identified **94 annotated acetyltransferases**. 

However, we know that UniProt can contain more up-to-date protein annotations than NCBI protein. Hence, we will also look into these protein annotations, to see if we can find any additional ones. *This was executed on UniProt release 2024_01, and not repeated here, as acetyltransferase annotations might have been added to / removed from UniProt since then, which could potentially affect downstream reuslts.*

In [ ]:
# function for getting protein sequence from UniProt
def fetch_sequence_uniprot(protein_id):
    url_fasta = "https://rest.uniprot.org/uniprotkb/{0}.fasta"
    # getting info
    fasta = sess.get(url_fasta.format(protein_id), stream = True).text
    seq = fasta.split("\n")
    return "".join(seq[1:-1])

In [ ]:
# url
url_acetyltransferase = "https://rest.uniprot.org/uniprotkb/search?query=organism_id:{0}+AND+protein_name:acetyltransferase&format=tsv"
url_gnat = "https://rest.uniprot.org/uniprotkb/search?query=organism_id:{0}+AND+protein_name:GNAT&format=tsv"
# empty df
acetyltransferases_up = pd.DataFrame()
# add in data
for phage in phages:
    taxid = phage.split("taxid_")[1]
    acetyltransferase_overview = pd.read_csv(StringIO(sess.get(url_acetyltransferase.format(taxid), stream = True).text), sep = "\t")
    gnat_overview = pd.read_csv(StringIO(sess.get(url_gnat.format(taxid), stream = True).text), sep = "\t")
    if acetyltransferase_overview.shape[0] > 0:
        acetyltransferase_overview = acetyltransferase_overview.assign(name_taxid = phage)
        acetyltransferases_up = pd.concat([acetyltransferases_up, acetyltransferase_overview], ignore_index = True)
    if gnat_overview.shape[0] > 0:
        gnat_overview = gnat_overview.assign(name_taxid = phage)
        acetyltransferases_up = pd.concat([acetyltransferases_up, gnat_overview], ignore_index = True)
# remove entries found by both queries
acetyltransferases_up = acetyltransferases_up.drop_duplicates(keep = "last").reset_index(drop = True) 
# add sequences 
acetyltransferases_up["sequence"] = acetyltransferases_up.apply(lambda row: fetch_sequence_uniprot(row["Entry"]), axis = 1)
# change dataframe to only keep relevant columns & rename columns in analogy to acetyltransferases_ncbi
acetyltransferases_up.drop(["Entry Name", "Reviewed", "Gene Names", "Organism"], axis = 1, inplace = True)
acetyltransferases_up = acetyltransferases_up.rename(columns = {"Entry": "uniprot_protein_id", 
                                                                "Protein names": "uniprot_protein_name", "Length": "uniprot_length"})

Our UniProt search identified **179 annotated acetyltransferases**. Let's combine the data to get an idea of the overlap:

In [ ]:
# match ncbi and uniprot acetyltransferases, remove duplicates
acetyltransferases = pd.merge(acetyltransferases_ncbi, acetyltransferases_up, how = "outer", on = ["name_taxid", "sequence"])
# add in taxonomic data phage & host
acetyltransferases = acetyltransferases.merge(phage_data, on = "name_taxid")
# remove duplicate columns 
    # deal with protein length - make 1 column (protein property, not database specific)
acetyltransferases["length"] = acetyltransferases["ncbi_protein_length"].fillna(acetyltransferases["uniprot_length"])
    # cast length to int
acetyltransferases["length"] = acetyltransferases["length"].astype("int")
    # remove duplicate length columns
acetyltransferases = acetyltransferases.drop(["ncbi_protein_length", "uniprot_length", "short_name", "host_name_string", "host_lineage_string"], axis=1)
# set correct datatypes - Int64 as we have NAs
acetyltransferases["ncbi_protein_uid"] = acetyltransferases["ncbi_protein_uid"].astype("Int64")
# let's reorder the columns to group protein data, phage & host data, uniprot & ncbi data
acetyltransferases_organized = acetyltransferases[["sequence", "length", 
                                                   "virus_name", "virus_taxid", "name_taxid", "virus_lineage",
                                                   "host_name", "host_lineage", 
                                                   "ncbi_protein_uid", "ncbi_protein_name", 
                                                   "uniprot_protein_id", "uniprot_protein_name"]]
# store to file
acetyltransferases_organized.to_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", "annotated_acetyltransferases_overview.tsv"), 
                                    sep = "\t", index = False)

Let's read in the data as obtained in February 2024, and use it for the rest of notebook: 

In [22]:
# read in the data as obtained in Febraury 2024
acetyltransferases = pd.read_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", "annotated_acetyltransferases_overview.tsv"), sep = "\t",)

In [23]:
# some statistics
    # number of annotated proteins
print(f"Found {len(acetyltransferases)} annotated acetyltransferases.")
    # number of included phages
num_included_phages = len(acetyltransferases["name_taxid"].unique())
print(f"These were found in {num_included_phages} different phages.")
    # number of phages with more than 1 annotated acetyltransferase
acetyltransferases_by_phage = acetyltransferases.groupby(["name_taxid"], as_index = False).size()
num_multiple = len(acetyltransferases_by_phage[acetyltransferases_by_phage["size"] > 1])
print(f"There were {num_multiple} phages with multiple annotated acetyltransferase.")

Found 213 annotated acetyltransferases.
These were found in 157 different phages.
There were 44 phages with multiple annotated acetyltransferase.


From this table, we can get a first glimpse into the annotated acetyltransferases (size, target, taxonomy). 

#### Goal 5: extract all sequences that will be used in the acetyltransferase search

To make sure we identify as many potential acetyltransferases as possible, we will also look at unannotated phage proteins. We will limit ourselves to proteins between 100-850 amino acids, which is a wide range around the expected size of GCN5-related N-acetyltransferases, the only protein acetyltransferase family reported in bacteria (this family is found in all domains of life, and contains the 2 known phage-encoded acetyltransferases).

As such, we wish to extract any phage protein from our NCBI protein overview that does not have any known function, and is within 100-850 amino acids in size. Most commonly, proteins of unknown function are annotated as 'hypothetical protein' or 'phage protein'. Hence, we will extract these proteins. We will also deal with duplicate NCBI protein entries by removing duplicate sequence entries within one phage.

*This was executed in February 2024, and not repeated here. In theory, a rerun should reproduce the original files (although sometimes instead of erroring, no results are returned upon API call), but it takes approximately 8 hours to rerun. As a result, the full output (detailing which phages have no hypothetical proteins  - within size limits - ) is not printed here, but simply added in the comments.*

In [ ]:
# loop over phages
for phage in phages:
    # deal with phage names
    if phage in annoying_phage_names_dict.keys():
        f_per_phage = os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{annoying_phage_names_dict.get(phage)}_protein_overview.tsv")
        out_per_phage = os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{annoying_phage_names_dict.get(phage)}_input_search.tsv")
    else:    
        f_per_phage = os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_protein_overview.tsv")
        out_per_phage = os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_input_search.tsv")
    # read in phage protein data fetched earlier from NCBI
    protein_per_phage = pd.read_csv(f_per_phage, sep = "\t")
    # filter for hypothetical / phage protein
    unannotated_per_phage = protein_per_phage[(protein_per_phage["ncbi_protein_name"].str.contains("hypothetical protein", case = False)) | 
        (protein_per_phage["ncbi_protein_name"].str.contains("phage protein", case = False))]
    if unannotated_per_phage.shape[0] > 0:
        # keep only those between 100-850AA
        unannotated_per_phage_sizerange = unannotated_per_phage[(unannotated_per_phage["ncbi_protein_length"] >= 100) 
            & (unannotated_per_phage["ncbi_protein_length"] <= 850)].reset_index(drop = True) 
        # deduplicate entries based on sequence
        try:
            unannotated_per_phage_sizerange["sequence"] = unannotated_per_phage_sizerange.apply(lambda row: fetch_sequence_ncbi(row["ncbi_protein_uid"]), 
                                                                                                axis = 1)
            unannotated_per_phage_sizerange_dedup = unannotated_per_phage_sizerange.drop_duplicates(subset=["sequence"], 
                                                                                                    keep="last").reset_index(drop = True)  
        # store results 
            # if there are any proteins left of course after filtering
            if unannotated_per_phage_sizerange_dedup.shape[0] > 0:
                unannotated_per_phage_sizerange_dedup.to_csv(out_per_phage, sep = "\t", index = False)
            else: 
                # happens for phiCTX_taxid_35343 & pf8_ST274-AUS411_taxid_2686285
                print(f"No hypothetical proteins matching search criteria for phage {phage}.") 
        except requests.exceptions.RequestException:
            print(f"Error fetching a protein sequence for phage {phage}. Ommitted this phage for now.")
            continue
    else:
       # happens for Epa5_taxid_2719575, phi12_taxid_161736, phi13_taxid_134554, phi2954_taxid_593131, phiKZ_taxid_2905945, PP7_taxid_12023
           #  & PRR1_taxid_12024, SC_10_H1H2H8_2017_taxid_2723239, SC_8_H2H8_2017_taxid_2723240, SC_9_H1H8_2017_taxid_2723241, LeviOr01_taxid_2848094
       print(f"No hypothetical proteins for phage {phage}.") 

Based on the output, it seems that we have been able to capture proteins without known function for most of the phages. However, for 11 phages, no proteins matched the 'hypothetical protein' or 'phage protein' description. We will manually look into how their proteins are named to see if we can still include them:
- Epa5_taxid_2719575: has no proteins without annotated function
- **phi12_taxid_161736**: based on manual inspection, I would like to include the following proteins, with NCBI protein unique identifiers: 22855226, 22855212, 22855211, 15488105, 15488104 
- phi13_taxid_134554: has no proteins without annotated function
- **phi2954_taxid_593131**:  based on manual inspection, I would like to include all proteins (are called P1, P2, etc.)
- **phiKZ_taxid_2905945**:  based on manual inspection, I would like to include all proteins named PhiKZ followed by a number
- PP7_taxid_12023: has no proteins without annotated function
- PRR1_taxid_12024: has no proteins without annotated function
- SC_10_H1H2H8_2017_taxid_2723239: has no proteins without annotated function
- SC_8_H2H8_2017_taxid_2723240: has no proteins without annotated function
- SC_9_H1H8_2017_taxid_2723241: has no proteins without annotated function
- LeviOr01_taxid_2848094: has no proteins without annotated function

In [ ]:
# Phi12 addition
phage = "phi12_taxid_161736"
ids_phi12 = [22855226, 22855212, 22855211, 15488105, 15488104]
protein_overview_phi12 = pd.read_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_protein_overview.tsv"), sep = "\t")
unannotated_phi12 = protein_overview_phi12[protein_overview_phi12["ncbi_protein_uid"].isin(ids_phi12)].reset_index(drop = True) 
unannotated_phi12["sequence"] = unannotated_phi12.apply(lambda row: fetch_sequence_ncbi(row["ncbi_protein_uid"]), axis = 1)
unannotated_phi12_dedup = unannotated_phi12.drop_duplicates(subset = ["sequence"], keep = "last").reset_index(drop = True)  
if unannotated_phi12_dedup.shape[0] > 0:
    unannotated_phi12_dedup.to_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_input_search.tsv"), sep="\t", index=False)
else:
    print(f"No hypothetical proteins matching search criteria for phage {phage}.")
# Phi2954 addition
phage = "phi2954_taxid_593131"
protein_overview_phi2954 = pd.read_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_protein_overview.tsv"), sep = "\t")
unannotated_phi2954 = protein_overview_phi2954[(protein_overview_phi2954["ncbi_protein_length"] >= 100) & 
    (protein_overview_phi2954["ncbi_protein_length"] <= 850)].reset_index(drop = True) 
unannotated_phi2954["sequence"] = unannotated_phi2954.apply(lambda row: fetch_sequence_ncbi(row["ncbi_protein_uid"]), axis = 1)
unannotated_phi2954_dedup = unannotated_phi2954.drop_duplicates(subset = ["sequence"], keep = "last").reset_index(drop = True)  
if unannotated_phi2954_dedup.shape[0] > 0:
    unannotated_phi2954_dedup.to_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_input_search.tsv"), 
                                     sep = "\t", index = False)
else:
    print(f"No hypothetical proteins matching search criteria for phage {phage}.")
# PhiKZ addition
phage = "phiKZ_taxid_2905945"
protein_overview_phikz = pd.read_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_protein_overview.tsv"), sep = "\t")
unannotated_phikz = protein_overview_phikz[(protein_overview_phikz["ncbi_protein_length"] >= 100) & 
    (protein_overview_phikz["ncbi_protein_length"] <= 850)].reset_index(drop = True)  
unannotated_phikz[["protein_name_split", "species"]] = unannotated_phikz["ncbi_protein_name"].str.split("[", expand = True)
unannotated_phikz = unannotated_phikz[(unannotated_phikz["species"].str.contains("Pseudomonas phage phiKZ]")) 
    & (unannotated_phikz["protein_name_split"].str.contains("PhiKZ", case = False))].reset_index(drop = True)  
unannotated_phikz["sequence"] = unannotated_phikz.apply(lambda row: fetch_sequence_ncbi(row["ncbi_protein_uid"]), axis = 1)
unannotated_phikz_dedup = unannotated_phikz.drop_duplicates(subset = ["sequence"], keep = "last").reset_index(drop = True)  
unannotated_phikz_dedup = unannotated_phikz_dedup.drop(columns = ["protein_name_split", "species"])
if unannotated_phikz_dedup.shape[0] > 0:
    unannotated_phikz_dedup.to_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_input_search.tsv"), 
                                   sep = "\t", index = False)
else:
    print(f"No hypothetical proteins matching search criteria for phage {phage}.")

Now let's read in the data from Febrary 2024 & combine all these proteins to see how many are out there:

In [24]:
unannotated_proteins = pd.DataFrame()
# add in data
for phage in phages:
    # deal with phage names
    if phage in annoying_phage_names_dict.keys():
        out_per_phage = os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{annoying_phage_names_dict.get(phage)}_input_search.tsv")
    else:    
        out_per_phage = os.path.join(pipeline_search_dir, "a_input", "protein_overview", f"{phage}_input_search.tsv")
    # read files, add in phage name for aggregation
    if os.path.isfile(out_per_phage):
        input_search = pd.read_csv(out_per_phage, sep = "\t")
        input_search = input_search.assign(name_taxid = phage)
        input_search = input_search[["name_taxid", "ncbi_protein_uid", "ncbi_protein_name",	"ncbi_protein_length", "sequence"]]
        # combine all phage files
        unannotated_proteins = pd.concat([input_search, unannotated_proteins], ignore_index = True)
# create overview file
unannotated_proteins.to_csv(os.path.join(pipeline_search_dir, "a_input", "protein_overview", "input_search.tsv") , sep = "\t", index = False)

In [25]:
# some simple statistics
total_input = len(unannotated_proteins)
unique_input = len(unannotated_proteins.drop_duplicates(subset = ["sequence"], keep = "last"))
print(f"There are {total_input} proteins of unknown function, totalling {unique_input} unique sequences.")

There are 38189 proteins of unknown function, totalling 20874 unique sequences.
